# Transformer — Implementations

Attention, positional encoding, and the encoder block again — on tensors, and through torch's own modules where the comparison is exact. Everything here is a pure function of its inputs (no training loops), so with the weights drawn from the same numpy generator every lane computes on literally identical numbers and the equivalence deltas sit at machine precision.

## 16_scaled_dot_product_attention

Softmax-weighted lookup: matmul, scale, mask, softmax, matmul.

### torch

The same five steps on tensors — matmul, scale, mask, softmax, matmul. **What torch adds:** autograd — the identical forward code is one `.backward()` away from attention gradients, which the checks demonstrate.

In [ ]:
import numpy as np
import torch

# hints:
# 1. K.transpose(-2, -1) replaces K.T; torch.softmax(scores, dim=-1) replaces yours.
# 2. as_tensor(x, dtype=torch.float64) converts numpy and passes tensors through untouched.
# 3. The additive mask survives unchanged: -inf before softmax is exactly 0 after.
# 4. Scale by sqrt of the LAST dim of Q — d_k, not the sequence length.


def scaled_dot_product_attention(Q, K, V, mask=None):
    """Scaled dot-product attention on tensors — the same five steps.

    Accepts numpy arrays or tensors; returns float64 tensors with the numpy
    lane's shapes: output (n_q, d_v), attn_weights (n_q, n_k).
    """
    Q = torch.as_tensor(Q, dtype=torch.float64)
    K = torch.as_tensor(K, dtype=torch.float64)
    V = torch.as_tensor(V, dtype=torch.float64)
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / d_k ** 0.5
    if mask is not None:
        scores = scores + torch.as_tensor(mask, dtype=torch.float64)
    attn_weights = torch.softmax(scores, dim=-1)
    output = attn_weights @ V
    return output, attn_weights


In [ ]:
# exports: attn_out, attn_w, attn_out_masked, attn_w_masked
_rng_eq = np.random.default_rng(160)
Q_eq = _rng_eq.standard_normal((4, 6))
K_eq = _rng_eq.standard_normal((5, 6))
V_eq = _rng_eq.standard_normal((5, 3))
_mask_eq = np.where(np.arange(5)[None, :] > np.arange(4)[:, None], -np.inf, 0.0)

attn_out, attn_w = scaled_dot_product_attention(Q_eq, K_eq, V_eq)
attn_out_masked, attn_w_masked = scaled_dot_product_attention(Q_eq, K_eq, V_eq, _mask_eq)
print("output:", tuple(attn_out.shape), " weights:", tuple(attn_w.shape))
print("masked row 0 weights:", attn_w_masked[0].numpy().round(6))


In [ ]:
assert torch.allclose(attn_w.sum(dim=-1), torch.ones(4, dtype=torch.float64), atol=1e-12),     "softmax rows are probability distributions"
# Each output coordinate is a convex combination of the value rows.
_Vt = torch.as_tensor(V_eq)
assert bool(torch.all(attn_out <= _Vt.max(dim=0).values + 1e-12)), "output stays under the value maxima"
assert bool(torch.all(attn_out >= _Vt.min(dim=0).values - 1e-12)), "output stays above the value minima"
assert float(attn_w_masked.triu(1).abs().max()) == 0.0, "-inf mask zeroes exactly the future keys"
# What torch adds: the same forward is differentiable end to end.
_Qg = torch.as_tensor(Q_eq).clone().requires_grad_(True)
_og, _ = scaled_dot_product_attention(_Qg, K_eq, V_eq)
_og.sum().backward()
assert _Qg.grad is not None and bool(torch.all(torch.isfinite(_Qg.grad))), "autograd flows through attention"


### library

`F.scaled_dot_product_attention` is the kernel behind every production transformer, with the same `1/√d_k` default scale and the same additive-mask convention. **What the library adds:** a fused kernel that never materialises the `(n_q, n_k)` weight matrix — which is why it returns no weights and the wrapper recomputes them.

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

# hints:
# 1. F.scaled_dot_product_attention wants a batch dim: unsqueeze(0) in, squeeze(0) out.
# 2. Its default scale is 1/sqrt(E) of the query — exactly the notebook's sqrt(d_k).
# 3. A float attn_mask is added to the scores, so the 0/-inf convention drops in as is.
# 4. The fused kernel never materialises weights; recompute them if the caller wants them.


def scaled_dot_product_attention(Q, K, V, mask=None):
    """The production kernel, wrapped to keep the notebook's signature.

    F.scaled_dot_product_attention computes softmax(QKᵀ/√d_k + mask) V in one
    fused call and never materialises the (n_q, n_k) weight matrix — that is
    its whole point — so the weights are recomputed here for the return value.
    """
    Q = torch.as_tensor(Q, dtype=torch.float64)
    K = torch.as_tensor(K, dtype=torch.float64)
    V = torch.as_tensor(V, dtype=torch.float64)
    attn_mask = None if mask is None else torch.as_tensor(mask, dtype=torch.float64)
    output = F.scaled_dot_product_attention(
        Q.unsqueeze(0), K.unsqueeze(0), V.unsqueeze(0),
        attn_mask=None if attn_mask is None else attn_mask.unsqueeze(0),
    ).squeeze(0)
    scores = Q @ K.transpose(-2, -1) / Q.shape[-1] ** 0.5
    if attn_mask is not None:
        scores = scores + attn_mask
    attn_weights = torch.softmax(scores, dim=-1)
    return output, attn_weights


In [ ]:
# exports: attn_out, attn_w, attn_out_masked, attn_w_masked
_rng_eq = np.random.default_rng(160)
Q_eq = _rng_eq.standard_normal((4, 6))
K_eq = _rng_eq.standard_normal((5, 6))
V_eq = _rng_eq.standard_normal((5, 3))
_mask_eq = np.where(np.arange(5)[None, :] > np.arange(4)[:, None], -np.inf, 0.0)

attn_out, attn_w = scaled_dot_product_attention(Q_eq, K_eq, V_eq)
attn_out_masked, attn_w_masked = scaled_dot_product_attention(Q_eq, K_eq, V_eq, _mask_eq)
print("output:", tuple(attn_out.shape), " weights:", tuple(attn_w.shape))
print("masked row 0 weights:", attn_w_masked[0].numpy().round(6))


In [ ]:
# The fused kernel computes exactly softmax(QK^T/sqrt(d)) V — nothing more.
_Vt = torch.as_tensor(V_eq)
assert torch.allclose(attn_out, attn_w @ _Vt, atol=1e-12), "fused output equals weights @ V"
assert torch.allclose(attn_w.sum(dim=-1), torch.ones(4, dtype=torch.float64), atol=1e-12),     "rows sum to 1"
assert float(attn_w_masked.triu(1).abs().max()) == 0.0, "float -inf mask blocks exactly"
assert torch.allclose(attn_out_masked, attn_w_masked @ _Vt, atol=1e-12), "masked path agrees too"


## 16_multi_head_attention

The same attention h times in parallel, each head in its own subspace.

### torch

The scratch class with tensors holding the weights, drawn from the same numpy generator in the same order so both lanes hold identical parameters. **What torch adds:** batched linear algebra — `(n_heads, n, d_k)` tensors make the per-head Python loop disappear into one matmul.

In [ ]:
import numpy as np
import torch

# hints:
# 1. Draw W_Q, W_K, W_V, W_O from the numpy rng in that order, then as_tensor them.
# 2. reshape(n, n_heads, d_k).permute(1, 0, 2) is _split_heads; permute back to merge.
# 3. One batched matmul over the head axis replaces the per-head Python loop.
# 4. Scale by the d_k of ONE head; the (n, n) mask broadcasts over the head axis.


class MultiHeadAttention:
    """Multi-head attention on tensors, weights drawn from the SAME numpy
    generator in the SAME order as the scratch class, so both lanes hold
    literally identical parameters."""

    def __init__(self, d_model, n_heads, rng=None):
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads  # = d_v
        rng = rng or np.random.default_rng()

        # Xavier initialization, numpy draws -> float64 tensors.
        scale = np.sqrt(2.0 / (d_model + self.d_k))
        self.W_Q = torch.as_tensor(rng.normal(0, scale, (d_model, d_model)))
        self.W_K = torch.as_tensor(rng.normal(0, scale, (d_model, d_model)))
        self.W_V = torch.as_tensor(rng.normal(0, scale, (d_model, d_model)))
        self.W_O = torch.as_tensor(rng.normal(0, scale, (d_model, d_model)))

    def _split_heads(self, X):
        """Reshape (n, d_model) -> (n_heads, n, d_k)."""
        return X.reshape(X.shape[0], self.n_heads, self.d_k).permute(1, 0, 2)

    def _merge_heads(self, X):
        """Reshape (n_heads, n, d_k) -> (n, d_model)."""
        return X.permute(1, 0, 2).reshape(-1, self.d_model)

    def forward(self, X, mask=None):
        """Self-attention: all heads at once via batched matmul."""
        Xt = torch.as_tensor(X, dtype=torch.float64)
        Q_h = self._split_heads(Xt @ self.W_Q)  # (n_heads, n, d_k)
        K_h = self._split_heads(Xt @ self.W_K)
        V_h = self._split_heads(Xt @ self.W_V)

        scores = Q_h @ K_h.transpose(-2, -1) / self.d_k ** 0.5
        if mask is not None:
            scores = scores + torch.as_tensor(mask, dtype=torch.float64)
        attn_weights = torch.softmax(scores, dim=-1)  # (n_heads, n, n)

        output = self._merge_heads(attn_weights @ V_h) @ self.W_O
        return output, attn_weights


In [ ]:
# exports: mha_out, mha_w, mha_out_masked
_mha_eq = MultiHeadAttention(16, 4, rng=np.random.default_rng(161))
X_mha_eq = np.random.default_rng(162).standard_normal((5, 16))
_mask_eq = np.where(np.arange(5)[None, :] > np.arange(5)[:, None], -np.inf, 0.0)

mha_out, mha_w = _mha_eq.forward(X_mha_eq)
mha_out_masked, _mw_eq = _mha_eq.forward(X_mha_eq, _mask_eq)
print("output:", tuple(mha_out.shape), " weights:", tuple(mha_w.shape))


In [ ]:
assert mha_out.shape == (5, 16) and mha_w.shape == (4, 5, 5)
assert torch.allclose(mha_w.sum(dim=-1), torch.ones(4, 5, dtype=torch.float64), atol=1e-12),     "every head's rows are distributions"
_Z = torch.as_tensor(np.random.default_rng(9).standard_normal((5, 16)))
assert torch.equal(_mha_eq._merge_heads(_mha_eq._split_heads(_Z)), _Z), "merge inverts split"
_om, _ow = _mha_eq.forward(X_mha_eq, _mask_eq)
assert float(_ow.triu(1).abs().max()) == 0.0, "the causal mask silences the future in every head"


### library

`nn.MultiheadAttention` with the scratch draws packed into `in_proj_weight`: q, k, v blocks stacked along dim 0, each transposed because torch linears compute `x @ Wᵀ` where the scratch class computes `x @ W`. **What the library adds:** the packing conventions — get the stack order or a transpose wrong and the output is garbage with a perfectly correct shape.

In [ ]:
import numpy as np
import torch
import torch.nn as nn

# hints:
# 1. in_proj_weight stacks the q, k, v projections along dim 0, in that order.
# 2. torch linears compute x @ W.T; the scratch class computes x @ W — transpose each block.
# 3. average_attn_weights=False keeps the per-head (n_heads, n, n) weights.
# 4. batch_first=True and unsqueeze(0): batched 3D input avoids unbatched edge cases.


class MultiHeadAttention:
    """nn.MultiheadAttention holding the scratch weights.

    The same numpy draws are packed into torch's conventions: q, k, v blocks
    stacked along dim 0 of in_proj_weight, each transposed because torch
    linears compute x @ Wᵀ where the scratch class computes x @ W. bias=False
    because the scratch class has no biases at all."""

    def __init__(self, d_model, n_heads, rng=None):
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        rng = rng or np.random.default_rng()

        scale = np.sqrt(2.0 / (d_model + self.d_k))
        W_Q = rng.normal(0, scale, (d_model, d_model))
        W_K = rng.normal(0, scale, (d_model, d_model))
        W_V = rng.normal(0, scale, (d_model, d_model))
        W_O = rng.normal(0, scale, (d_model, d_model))

        self.attn = nn.MultiheadAttention(d_model, n_heads, bias=False,
                                          batch_first=True).double()
        with torch.no_grad():
            self.attn.in_proj_weight.copy_(
                torch.as_tensor(np.concatenate([W_Q.T, W_K.T, W_V.T], axis=0)))
            self.attn.out_proj.weight.copy_(torch.as_tensor(W_O.T))

    def forward(self, X, mask=None):
        Xt = torch.as_tensor(X, dtype=torch.float64).unsqueeze(0)  # (1, n, d_model)
        attn_mask = None if mask is None else torch.as_tensor(mask, dtype=torch.float64)
        out, w = self.attn(Xt, Xt, Xt, attn_mask=attn_mask,
                           need_weights=True, average_attn_weights=False)
        return out.squeeze(0).detach(), w.squeeze(0).detach()


In [ ]:
# exports: mha_out, mha_w, mha_out_masked
_mha_eq = MultiHeadAttention(16, 4, rng=np.random.default_rng(161))
X_mha_eq = np.random.default_rng(162).standard_normal((5, 16))
_mask_eq = np.where(np.arange(5)[None, :] > np.arange(5)[:, None], -np.inf, 0.0)

mha_out, mha_w = _mha_eq.forward(X_mha_eq)
mha_out_masked, _mw_eq = _mha_eq.forward(X_mha_eq, _mask_eq)
print("output:", tuple(mha_out.shape), " weights:", tuple(mha_w.shape))


In [ ]:
# Unpack in_proj_weight and redo the scratch math by hand on a fresh instance:
# if the packing (stack order + transposes) is right, the numbers match.
_m = MultiHeadAttention(4, 2, rng=np.random.default_rng(3))
_X = torch.as_tensor(np.random.default_rng(4).standard_normal((3, 4)))
_o, _w = _m.forward(_X)
_Wq, _Wk, _Wv = _m.attn.in_proj_weight.chunk(3, dim=0)
_q = (_X @ _Wq.T).reshape(3, 2, 2).permute(1, 0, 2)
_k = (_X @ _Wk.T).reshape(3, 2, 2).permute(1, 0, 2)
_v = (_X @ _Wv.T).reshape(3, 2, 2).permute(1, 0, 2)
_sw = torch.softmax(_q @ _k.transpose(-2, -1) / 2 ** 0.5, dim=-1)
_manual = (_sw @ _v).permute(1, 0, 2).reshape(3, 4) @ _m.attn.out_proj.weight.T
assert torch.allclose(_o, _manual, atol=1e-12), "packed projections reproduce the scratch math"
assert torch.allclose(_w, _sw, atol=1e-12), "per-head weights match the hand computation"
assert torch.allclose(mha_w.sum(dim=-1), torch.ones(4, 5, dtype=torch.float64), atol=1e-12),     "rows sum to 1 in every head"


## 16_positional_encoding

Position stamped into the embedding with sines and cosines.

No library lane: `torch.nn` ships no sinusoidal positional-encoding module — real models register exactly this table as a buffer by hand — so there is no standard estimator to compare against.

### torch

The same table built with tensor ops, float64 end to end, so the two lanes agree to the last bit. **What torch adds:** honestly, only dtype/device plumbing — this is a fixed, parameter-free function of position, which is exactly the lesson.

In [ ]:
import numpy as np
import torch

# hints:
# 1. pos is a (max_len, 1) float64 column; pos * freq broadcasts to (max_len, d/2).
# 2. 10000.0 ** t works on a tensor via __rpow__ — the numpy spelling carries over.
# 3. Even slice 0::2 takes sin, odd slice 1::2 takes cos, both from the same freq.
# 4. No parameters and no rng anywhere — two lanes agree to the last bit for free.


def positional_encoding(max_len, d_model):
    """Sinusoidal positional encoding on tensors (same table as the numpy lane).

    Returns:
        PE: float64 tensor, shape (max_len, d_model)
    """
    PE = torch.zeros(max_len, d_model, dtype=torch.float64)
    pos = torch.arange(max_len, dtype=torch.float64).unsqueeze(1)  # (max_len, 1)
    i = torch.arange(0, d_model, 2, dtype=torch.float64)  # even indices
    freq = 1.0 / (10000.0 ** (i / d_model))  # (d_model/2,)

    PE[:, 0::2] = torch.sin(pos * freq)  # even dimensions: sin
    PE[:, 1::2] = torch.cos(pos * freq)  # odd dimensions: cos
    return PE


In [ ]:
# exports: pe_table
pe_table = positional_encoding(12, 16)
print("PE table:", tuple(pe_table.shape))
print("row 0:", pe_table[0, :6].numpy().round(3), " row 3:", pe_table[3, :6].numpy().round(3))


In [ ]:
assert pe_table.shape == (12, 16)
assert float(pe_table.abs().max()) <= 1.0 + 1e-12, "sin/cos keep every entry in [-1, 1]"
assert torch.allclose(pe_table[:, 0::2] ** 2 + pe_table[:, 1::2] ** 2,
                      torch.ones(12, 8, dtype=torch.float64), atol=1e-12),     "each (sin, cos) pair lies on the unit circle"
assert torch.allclose(pe_table[0, 0::2], torch.zeros(8, dtype=torch.float64), atol=1e-15) and     torch.allclose(pe_table[0, 1::2], torch.ones(8, dtype=torch.float64), atol=1e-15),     "position 0 encodes as (0, 1, 0, 1, ...)"
_d = torch.cdist(pe_table, pe_table) + torch.eye(12, dtype=torch.float64)
assert bool(torch.all(_d > 0.01)), "distinct positions get distinct encodings"


## 16_encoder_block

Attention, FFN, residuals, and post-LN — the original encoder block.

### torch

The full post-LN block on tensors: batched multi-head attention, FFN, residuals, and a hand-rolled `layer_norm`. **What torch adds:** mostly a trap this lane documents — `torch.var` is unbiased by default, and `unbiased=False` is the one line separating exact agreement from a silent mismatch.

In [ ]:
import numpy as np
import torch

# hints:
# 1. torch.var is unbiased by default; pass unbiased=False or LayerNorm will not match.
# 2. Draw order is sacred: W_Q, W_K, W_V, W_O, then W1, W2, all from the one rng.
# 3. Post-LN placement: normalise AFTER each residual add, as the 2017 paper did.
# 4. Attention is batched over heads here — one softmax call covers all of them.


def layer_norm(x, gamma, beta, eps=1e-5):
    """LayerNorm across the last dim. unbiased=False matches numpy's var (ddof=0)."""
    mu = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, unbiased=False, keepdim=True)
    x_norm = (x - mu) / torch.sqrt(var + eps)
    return gamma * x_norm + beta


def relu(x):
    return torch.clamp(x, min=0.0)


class TransformerEncoderBlock:
    """Post-LN encoder block on tensors: attention -> add&norm -> FFN -> add&norm.

    Weights come from the SAME numpy generator in the SAME order as the scratch
    block — W_Q, W_K, W_V, W_O, then W1, W2 — so both lanes hold literally
    identical parameters."""

    def __init__(self, d_model, n_heads, d_ff=None, rng=None):
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.d_ff = d_ff or 4 * d_model
        rng = rng or np.random.default_rng()

        # Multi-head attention weights (Xavier, same draws as the scratch mha).
        scale = np.sqrt(2.0 / (d_model + self.d_k))
        self.W_Q = torch.as_tensor(rng.normal(0, scale, (d_model, d_model)))
        self.W_K = torch.as_tensor(rng.normal(0, scale, (d_model, d_model)))
        self.W_V = torch.as_tensor(rng.normal(0, scale, (d_model, d_model)))
        self.W_O = torch.as_tensor(rng.normal(0, scale, (d_model, d_model)))

        # FFN weights.
        scale_ff = np.sqrt(2.0 / (d_model + self.d_ff))
        self.W1 = torch.as_tensor(rng.normal(0, scale_ff, (d_model, self.d_ff)))
        self.b1 = torch.zeros(self.d_ff, dtype=torch.float64)
        self.W2 = torch.as_tensor(rng.normal(0, scale_ff, (self.d_ff, d_model)))
        self.b2 = torch.zeros(d_model, dtype=torch.float64)

        # LayerNorm parameters (identity init).
        self.gamma1 = torch.ones(d_model, dtype=torch.float64)
        self.beta1 = torch.zeros(d_model, dtype=torch.float64)
        self.gamma2 = torch.ones(d_model, dtype=torch.float64)
        self.beta2 = torch.zeros(d_model, dtype=torch.float64)

    def forward(self, X, mask=None):
        Xt = torch.as_tensor(X, dtype=torch.float64)
        n = Xt.shape[0]

        # Sub-layer 1: multi-head self-attention, batched over the head axis.
        Q_h = (Xt @ self.W_Q).reshape(n, self.n_heads, self.d_k).permute(1, 0, 2)
        K_h = (Xt @ self.W_K).reshape(n, self.n_heads, self.d_k).permute(1, 0, 2)
        V_h = (Xt @ self.W_V).reshape(n, self.n_heads, self.d_k).permute(1, 0, 2)
        scores = Q_h @ K_h.transpose(-2, -1) / self.d_k ** 0.5
        if mask is not None:
            scores = scores + torch.as_tensor(mask, dtype=torch.float64)
        attn_weights = torch.softmax(scores, dim=-1)  # (n_heads, n, n)
        concat = (attn_weights @ V_h).permute(1, 0, 2).reshape(n, self.d_model)
        attn_out = concat @ self.W_O
        Z1 = layer_norm(Xt + attn_out, self.gamma1, self.beta1)

        # Sub-layer 2: FFN + residual + LayerNorm.
        ffn_out = relu(Z1 @ self.W1 + self.b1) @ self.W2 + self.b2
        Z2 = layer_norm(Z1 + ffn_out, self.gamma2, self.beta2)
        return Z2, attn_weights


In [ ]:
# exports: enc_out, enc_w
_enc_eq = TransformerEncoderBlock(16, 4, d_ff=32, rng=np.random.default_rng(163))
X_enc_eq = np.random.default_rng(164).standard_normal((6, 16))

enc_out, enc_w = _enc_eq.forward(X_enc_eq)
print("output:", tuple(enc_out.shape), " weights:", tuple(enc_w.shape))
print("row means:", enc_out.mean(dim=-1).numpy().round(5))


In [ ]:
assert enc_out.shape == (6, 16) and enc_w.shape == (4, 6, 6)
assert torch.allclose(enc_out.mean(dim=-1), torch.zeros(6, dtype=torch.float64), atol=1e-4),     "post-LN rows are centred"
assert bool(torch.all((enc_out.std(dim=-1, unbiased=False) - 1.0).abs() < 0.05)),     "post-LN rows have unit scale"
_g = torch.ones(16, dtype=torch.float64)
_b = torch.zeros(16, dtype=torch.float64)
_x = torch.as_tensor(X_enc_eq)
assert torch.allclose(layer_norm(_x + 3.0, _g, _b), layer_norm(_x, _g, _b), atol=1e-12),     "layer_norm ignores a per-row constant shift"
assert torch.allclose(layer_norm(_x, _g, _b),
                      torch.nn.functional.layer_norm(_x, (16,), _g, _b, 1e-5), atol=1e-12),     "the hand-rolled layer_norm matches torch's own"


### library

The scratch block is post-LN, and `norm_first=False` (the default) makes `nn.TransformerEncoderLayer` the same composition with matching `eps=1e-5` and ReLU — so the layer is exactly comparable once `dropout=0.0` and every weight is copied (transposed) from the same numpy draws. **What the library adds:** one module for the whole block — plus four biases the scratch block never had, all zeroed here.

In [ ]:
import numpy as np
import torch
import torch.nn as nn

# hints:
# 1. norm_first=False is post-LN — the same placement as the scratch block.
# 2. dropout=0.0 everywhere, or the answer changes between train() and eval().
# 3. Copy W.T into every torch Linear and zero every bias the scratch block lacks.
# 4. layer_norm_eps already defaults to the scratch 1e-5 — one convention agrees free.


class TransformerEncoderBlock:
    """nn.TransformerEncoderLayer holding the scratch weights.

    The scratch block is post-LN and norm_first=False (torch's default) is the
    same composition — norm(x + attn(x)), then norm(x + ffn(x)) — with matching
    eps=1e-5 and ReLU, so the layer is exactly comparable. The layer never
    returns attention weights, so forward asks its self_attn module separately."""

    def __init__(self, d_model, n_heads, d_ff=None, rng=None):
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_ff = d_ff or 4 * d_model
        d_k = d_model // n_heads
        rng = rng or np.random.default_rng()

        # Same draw order as the scratch block: W_Q, W_K, W_V, W_O, then W1, W2.
        scale = np.sqrt(2.0 / (d_model + d_k))
        W_Q = rng.normal(0, scale, (d_model, d_model))
        W_K = rng.normal(0, scale, (d_model, d_model))
        W_V = rng.normal(0, scale, (d_model, d_model))
        W_O = rng.normal(0, scale, (d_model, d_model))
        scale_ff = np.sqrt(2.0 / (d_model + self.d_ff))
        W1 = rng.normal(0, scale_ff, (d_model, self.d_ff))
        W2 = rng.normal(0, scale_ff, (self.d_ff, d_model))

        self.layer = nn.TransformerEncoderLayer(
            d_model, n_heads, dim_feedforward=self.d_ff, dropout=0.0,
            activation="relu", batch_first=True, norm_first=False).double()
        with torch.no_grad():
            self.layer.self_attn.in_proj_weight.copy_(
                torch.as_tensor(np.concatenate([W_Q.T, W_K.T, W_V.T], axis=0)))
            self.layer.self_attn.in_proj_bias.zero_()
            self.layer.self_attn.out_proj.weight.copy_(torch.as_tensor(W_O.T))
            self.layer.self_attn.out_proj.bias.zero_()
            self.layer.linear1.weight.copy_(torch.as_tensor(W1.T))
            self.layer.linear1.bias.zero_()
            self.layer.linear2.weight.copy_(torch.as_tensor(W2.T))
            self.layer.linear2.bias.zero_()
            # norm1/norm2 default to gamma=1, beta=0 — exactly the scratch init.

    def forward(self, X, mask=None):
        Xt = torch.as_tensor(X, dtype=torch.float64).unsqueeze(0)  # (1, n, d_model)
        attn_mask = None if mask is None else torch.as_tensor(mask, dtype=torch.float64)
        out = self.layer(Xt, src_mask=attn_mask).squeeze(0).detach()
        _, w = self.layer.self_attn(Xt, Xt, Xt, attn_mask=attn_mask,
                                    need_weights=True, average_attn_weights=False)
        return out, w.squeeze(0).detach()


In [ ]:
# exports: enc_out, enc_w
_enc_eq = TransformerEncoderBlock(16, 4, d_ff=32, rng=np.random.default_rng(163))
X_enc_eq = np.random.default_rng(164).standard_normal((6, 16))

enc_out, enc_w = _enc_eq.forward(X_enc_eq)
print("output:", tuple(enc_out.shape), " weights:", tuple(enc_w.shape))
print("row means:", enc_out.mean(dim=-1).numpy().round(5))


In [ ]:
# Recompose the block from the layer's own submodules: if the placement is
# post-LN, norm2(z1 + ffn(z1)) with z1 = norm1(x + attn(x)) must reproduce it.
_Xt = torch.as_tensor(X_enc_eq).unsqueeze(0)
_a, _ = _enc_eq.layer.self_attn(_Xt, _Xt, _Xt, need_weights=False)
_z1 = _enc_eq.layer.norm1(_Xt + _a)
_f = _enc_eq.layer.linear2(torch.relu(_enc_eq.layer.linear1(_z1)))
_z2 = _enc_eq.layer.norm2(_z1 + _f)
assert torch.allclose(_z2.squeeze(0), enc_out, atol=1e-12),     "post-LN composition reproduces the layer output"
assert torch.allclose(enc_w.sum(dim=-1), torch.ones(4, 6, dtype=torch.float64), atol=1e-12),     "attention rows sum to 1 in every head"
assert torch.allclose(enc_out.mean(dim=-1), torch.zeros(6, dtype=torch.float64), atol=1e-4),     "rows are centred — the signature of a final LayerNorm (post-LN)"
